Intermediate: Macroeconomic flows to and from Canada
====================================================

Flow types included:
- Imports
- Exports
- Foreign direct investment by immediate investor
- Foreign direct investment by ultimate investor

In [2]:
from time import sleep
from pathlib import Path 
import pandas as pd 
import sys

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from utils.paths import (STG_STATCAN_DIR, STG_GAC_DIR, STG_UN_DIR,
                         INT_MACROECON_FLOWS_DIR)
from utils.etl_utils import merge_columns_safe

In [4]:
# Load the staged economic data
stg_imports_df = pd.read_csv(STG_STATCAN_DIR / 'stg_statcan__imports.csv')
stg_exports_df = pd.read_csv(STG_STATCAN_DIR / 'stg_statcan__exports.csv')
stg_fdi_immediate_df = pd.read_csv(STG_STATCAN_DIR / 'stg_statcan__fdi_immediate.csv')
stg_fdi_ultimate_df  = pd.read_csv(STG_STATCAN_DIR / 'stg_statcan__fdi_ultimate.csv')



In [5]:
# Imports: rename to a common "region" column and label the flow type
imports_final = (
    stg_imports_df
    .assign(flow_type="Imports from region")
    .rename(columns={"imports_from_region": "region"})
)

# Exports: same idea
exports_final = (
    stg_exports_df
    .assign(flow_type="Exports to region")
    .rename(columns={"exports_to_region": "region"})
)

In [6]:
fdi_type_map = {
    # FDI by immediate investor
    "Canadian direct investment abroad - total book value":
        "FDI immediate outward, total book value",
    "Foreign direct investment in Canada - total book value":
        "FDI immediate inward, total book value",

    # FDI by ultimate investor
    "Canadian direct investment abroad by ultimate investor country – total book value":
        "FDI ultimate outward, total book value",
    "Foreign direct investment in Canada by ultimate investor country - total book value":
        "FDI ultimate inward, total book value",
}

In [7]:
# Immediate FDI: create flow_type from fdi_type
fdi_immediate_final = (
    stg_fdi_immediate_df
    .assign(flow_type=stg_fdi_immediate_df["fdi_type"].map(fdi_type_map))
    .drop(columns=["fdi_type"])
)

# Ultimate FDI
fdi_ultimate_final = (
    stg_fdi_ultimate_df
    .assign(flow_type=stg_fdi_ultimate_df["fdi_type"].map(fdi_type_map))
    .drop(columns=["fdi_type"])
)

In [8]:
# Concatenate all macro flows into a single intermediate table

# We assume each DF has: year, region (or renamed to region),
# dollar_value, and now flow_type
int_macroecon_flows = pd.concat(
    [imports_final, exports_final, fdi_immediate_final, fdi_ultimate_final],
    ignore_index=True,
)

# Keep just the core columns
col_order = ["year", "region", "flow_type", "dollar_value"]

int_macroecon_flows = (
    int_macroecon_flows[col_order]
    .drop_duplicates(ignore_index=True)
    .convert_dtypes()
)

In [11]:
int_macroecon_flows


,year,region,flow_type,dollar_value
0,2005,All countries,Imports from region,363851094000
1,2005,Europe,Imports from region,49219344000
2,2005,Belgium,Imports from region,2612159000
3,2005,Bulgaria,Imports from region,63778000
4,2005,Czechia,Imports from region,163572000
...,...,...,...,...
24624,2024,Singapore,"FDI ultimate outward, total book value",-373000000
24625,2024,United Arab Emirates,"FDI ultimate outward, total book value",3204000000
24626,2024,Other Asia/Oceania,"FDI ultimate outward, total book value",3238000000
24627,2024,Canada,"FDI ultimate outward, total book value",2294897000000


In [9]:
# Save to intermediate flows
filename = "int_macroecon_flows.csv"
filepath = INT_MACROECON_FLOWS_DIR / filename

int_macroecon_flows.to_csv(filepath, index=False)